# Mat & Comp – Lección 10: Probabilidad
## El problema de Monty Hall y la paradoja del cumpleaños

**Curso:** Matemáticas y Computación · **Duración sugerida:** 2 sesiones de 50 minutos

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Calcular probabilidades **contando casos** (favorables / posibles) y comprobarlas con **simulaciones**.
2. Resolver el **problema de Monty Hall** de tres formas: jugando, simulando y enumerando todos los casos.
3. Entender cómo la **información nueva** cambia una probabilidad (y por qué importa lo que *sabe* quien da esa información).
4. Calcular la probabilidad de que dos personas de un grupo **cumplan años el mismo día** usando el complemento.
5. Desconfiar de la intuición en probabilidad y usar la computadora para ponerla a prueba.

## 🔍 ¿Cambiarías de puerta?

Estás en un concurso de televisión. Hay **tres puertas**: detrás de una hay un **carro** 🚗 y detrás de las otras dos, una **cabra** 🐐.

1. Eliges una puerta, por ejemplo la 1.
2. El presentador, **que sabe dónde está el carro**, abre otra puerta, por ejemplo la 3, y muestra una cabra.
3. Te pregunta: **¿quieres quedarte con la puerta 1 o cambiarte a la 2?**

✏️ *¿Qué harías? ¿Da lo mismo? Escribe tu respuesta y tu razón antes de seguir.*

En 1990 la columnista Marilyn vos Savant respondió en una revista que **conviene cambiar**. Recibió unas 10 000 cartas diciéndole que estaba equivocada… muchas de ellas de profesores de matemáticas.

## 📚 El concepto: probabilidad clásica y frecuencias

Si todos los resultados de un experimento son **igual de probables**:

$$P(\text{evento}) = \frac{\text{casos favorables}}{\text{casos posibles}}$$

En la Lección 2 vimos la otra cara: si repetimos un experimento muchas veces, la **frecuencia** con que ocurre algo se acerca a su probabilidad (ley de los grandes números).

Hoy usaremos las dos ideas juntas: **contar** para obtener la respuesta exacta y **simular** para comprobarla… o para descubrirla cuando contar es difícil.

## 1. Juega una ronda

In [ ]:
import random
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, ToggleButtons, IntSlider

def jugar(eleccion, cambiar, rng=random):
    """Juega una ronda de Monty Hall. Devuelve True si gana el carro."""
    puertas = [1, 2, 3]
    carro = rng.choice(puertas)
    # el presentador abre una puerta que no es la tuya y que tiene cabra
    opciones = [p for p in puertas if p != eleccion and p != carro]
    abierta = rng.choice(opciones)
    if cambiar:
        eleccion = [p for p in puertas if p != eleccion and p != abierta][0]
    return eleccion == carro, carro, abierta, eleccion

def ronda(numero_de_ronda, tu_puerta, estrategia):
    rng = random.Random(numero_de_ronda)       # cada número de ronda es un juego distinto
    gano, carro, abierta, final = jugar(tu_puerta, estrategia == "Cambiar", rng)
    print(f"Elegiste la puerta {tu_puerta}. El presentador abre la puerta {abierta}: ¡cabra! 🐐")
    print(f"Decides {'cambiar' if estrategia == 'Cambiar' else 'quedarte'}: tu puerta final es la {final}.")
    print(f"El carro estaba en la puerta {carro}.", "¡GANASTE! 🚗" if gano else "Perdiste... 🐐")

interact(ronda,
         numero_de_ronda=IntSlider(value=1, min=1, max=50),
         tu_puerta=Dropdown(options=[1, 2, 3], value=1),
         estrategia=ToggleButtons(options=["Quedarme", "Cambiar"]));

Juega **20 rondas** con cada estrategia (cambia el número de ronda con el slider) y anota cuántas ganas. ¿Notas alguna diferencia? Con 20 rondas todavía puede haber mucha suerte… así que pongamos a la computadora a jugar miles de veces.

## 2. Simular miles de juegos

In [ ]:
def porcentaje_de_victorias(cambiar, juegos):
    ganadas = sum(jugar(random.randint(1, 3), cambiar)[0] for _ in range(juegos))
    return ganadas / juegos

random.seed(10)
print("Quedarse:", porcentaje_de_victorias(False, 100000))
print("Cambiar: ", porcentaje_de_victorias(True, 100000))

In [ ]:
# La proporción acumulada de victorias a lo largo de 3000 juegos
def acumulado(cambiar, juegos):
    ganadas, lista = 0, []
    for i in range(1, juegos + 1):
        ganadas += jugar(random.randint(1, 3), cambiar)[0]
        lista.append(ganadas / i)
    return lista

plt.figure(figsize=(9, 4))
plt.plot(acumulado(False, 3000), label="quedarse")
plt.plot(acumulado(True, 3000), label="cambiar")
plt.axhline(1 / 3, ls="--", color="gray"); plt.axhline(2 / 3, ls="--", color="gray")
plt.ylim(0, 1); plt.xlabel("juegos"); plt.ylabel("proporción de victorias")
plt.title("Monty Hall: ¿quedarse o cambiar?")
plt.legend(); plt.show()

**Quedarse gana 1/3 de las veces y cambiar gana 2/3.** ¡Cambiar duplica tus posibilidades!

## 3. ¿Por qué? Contemos todos los casos

Supón que siempre eliges la puerta 1. El carro puede estar en cualquiera de las tres, con la misma probabilidad:

| Carro en | Presentador abre | Si te quedas | Si cambias |
|---|---|---|---|
| 1 | 2 o 3 | ✅ gana | ❌ pierde |
| 2 | 3 (obligado) | ❌ pierde | ✅ gana |
| 3 | 2 (obligado) | ❌ pierde | ✅ gana |

**La clave:** al principio tu puerta tiene 1/3 de probabilidad y las otras dos juntas tienen 2/3. El presentador **no puede** abrir tu puerta ni la del carro, así que al abrir una cabra "concentra" los 2/3 en la puerta que queda cerrada.

In [ ]:
# Enumeramos todos los casos posibles (sin azar) con la puerta 1 elegida
gana_quedarse, gana_cambiar = 0, 0
for carro in [1, 2, 3]:
    gana_quedarse += (carro == 1)
    gana_cambiar += (carro != 1)
print(f"Quedarse gana en {gana_quedarse} de 3 casos; cambiar gana en {gana_cambiar} de 3 casos")

### Con muchas puertas se ve más claro

Imagina **100 puertas**. Eliges una. El presentador abre **98** puertas con cabra y deja una cerrada. ¿Te quedas con la tuya, que elegiste a ciegas entre 100?

In [ ]:
def jugar_n(n_puertas, cambiar):
    carro = random.randint(1, n_puertas)
    eleccion = random.randint(1, n_puertas)
    # el presentador deja cerrada solo una puerta además de la tuya:
    # la del carro (si no la elegiste) o una al azar (si la elegiste)
    if eleccion == carro:
        otra = random.choice([p for p in range(1, n_puertas + 1) if p != eleccion])
    else:
        otra = carro
    final = otra if cambiar else eleccion
    return final == carro

def ver_n_puertas(n_puertas):
    juegos = 20000
    q = sum(jugar_n(n_puertas, False) for _ in range(juegos)) / juegos
    c = sum(jugar_n(n_puertas, True) for _ in range(juegos)) / juegos
    plt.figure(figsize=(6, 3))
    plt.bar(["quedarse", "cambiar"], [q, c], color=["#8d99ae", "#2a9d8f"])
    plt.ylim(0, 1); plt.title(f"{n_puertas} puertas ({juegos} juegos)")
    plt.show()
    print(f"Quedarse: {q:.3f} (teoría 1/{n_puertas} = {1 / n_puertas:.3f})")
    print(f"Cambiar:  {c:.3f} (teoría {n_puertas - 1}/{n_puertas} = {(n_puertas - 1) / n_puertas:.3f})")

interact(ver_n_puertas, n_puertas=IntSlider(value=3, min=3, max=100, continuous_update=False));

**¿Qué pasa si…?**
- …el presentador **no sabe** dónde está el carro y abre una de las otras puertas al azar? A veces mostrará el carro (y el juego se anula). Si muestra una cabra, ¿sigue conviniendo cambiar? Simúlalo en la celda de abajo.

In [ ]:
# Monty "despistado": abre al azar una de las otras dos puertas
def jugar_despistado(cambiar):
    carro = random.randint(1, 3)
    eleccion = 1
    abierta = random.choice([2, 3])
    if abierta == carro:
        return None                       # mostró el carro: el juego no cuenta
    final = ({2, 3} - {abierta}).pop() if cambiar else eleccion
    return final == carro

for cambiar in [False, True]:
    resultados = [jugar_despistado(cambiar) for _ in range(100000)]
    validos = [r for r in resultados if r is not None]
    print("Cambiar" if cambiar else "Quedarse", "→", round(sum(validos) / len(validos), 3))

¡Ahora da **lo mismo**: 1/2 y 1/2! En el juego original el presentador nos da información porque **sabe** y **elige** con cuidado. Cuando abre al azar, mostrar una cabra no nos dice nada nuevo sobre nuestra puerta. En probabilidad importa **cómo** se obtiene la información, no solo cuál es.

## 4. La paradoja del cumpleaños

**¿Cuántas personas debe haber en un salón para que sea más probable que no que dos de ellas cumplan años el mismo día?**

✏️ *Escribe tu predicción. (Muchas personas dicen 183, la mitad de 365.)*

Calcular directamente "al menos dos coinciden" es difícil. Es más fácil calcular lo **contrario** (el **complemento**): que **todas** cumplan años en días distintos.

- La primera persona puede cumplir cualquier día: 365/365.
- La segunda debe evitar ese día: 364/365.
- La tercera debe evitar dos días: 363/365… y así.

$$P(\text{todos distintos}) = \frac{365}{365} \cdot \frac{364}{365} \cdots \frac{365 - n + 1}{365}, \qquad P(\text{coincidencia}) = 1 - P(\text{todos distintos})$$

In [ ]:
def prob_coincidencia(n):
    distintos = 1.0
    for k in range(n):
        distintos *= (365 - k) / 365
    return 1 - distintos

for n in [5, 10, 20, 22, 23, 30, 40, 50, 70]:
    print(f"{n:3} personas → {prob_coincidencia(n):6.1%}")

In [ ]:
# Comprobamos con una simulación: grupos de n personas con cumpleaños al azar
def simular_cumple(n, grupos=20000):
    exitos = 0
    for _ in range(grupos):
        cumples = [random.randint(1, 365) for _ in range(n)]
        if len(set(cumples)) < n:        # set quita repetidos: si se encoge, hubo coincidencia
            exitos += 1
    return exitos / grupos

ns = list(range(1, 71))
plt.figure(figsize=(9, 4))
plt.plot(ns, [prob_coincidencia(n) for n in ns], label="fórmula")
plt.scatter(ns[::3], [simular_cumple(n, 3000) for n in ns[::3]], color="crimson", s=15, label="simulación")
plt.axhline(0.5, ls="--", color="gray"); plt.axvline(23, ls=":", color="gray")
plt.xlabel("personas en el salón"); plt.ylabel("P(al menos dos cumplen el mismo día)")
plt.legend(); plt.title("La paradoja del cumpleaños")
plt.show()

Con solo **23 personas** la probabilidad ya supera el 50 %, y con 70 es casi segura (99,9 %).

¿Por qué es tan sorprendente? Pensamos en "alguien que cumpla **el mismo día que yo**", pero el problema habla de **cualquier pareja**. Con 23 personas hay 23 · 22 / 2 = **253 parejas** distintas, ¡y cada una es una oportunidad de coincidir! (¿Recuerdas los números triangulares de la Lección 1?)

**¿Qué pasa si…?**
- …calculas la probabilidad de que alguien del salón cumpla años **el mismo día que tú**? ¿Cuántas personas hacen falta para superar el 50 %? (Pista: 1 − (364/365)ⁿ.)

---
## ✍️ Ejercicios

Resuelve cada ejercicio en la celda de código que está debajo. Para cada uno, calcula la respuesta **exacta** (contando casos) cuando puedas y **compruébala** con una simulación.

### Ejercicio 1 (básico) · Dos dados
Se lanzan dos dados.
a) Recorre las 36 parejas posibles con dos `for` y cuenta en cuántas la suma es 7. ¿Cuál es la probabilidad?
b) ¿Cuál es la probabilidad de sacar **al menos un 6**? (Pista: complemento.)
c) Comprueba ambas con 100 000 lanzamientos simulados.

In [ ]:
# Escribe tu código aquí

### Ejercicio 2 (básico) · Tu salón
a) ¿Cuál es la probabilidad de que en tu salón haya dos personas con el mismo cumpleaños? Usa `prob_coincidencia` con el número de estudiantes.
b) Hagan la prueba de verdad: ¿hay alguna coincidencia?

In [ ]:
# Escribe tu código aquí

### Ejercicio 3 (intermedio) · Las tres cartas
En una bolsa hay tres cartas: una **roja por ambos lados**, una **blanca por ambos lados** y una **roja por un lado y blanca por el otro**. Sacas una carta al azar y la pones sobre la mesa: la cara de arriba es **roja**.
¿Cuál es la probabilidad de que la otra cara también sea roja? Muchos dicen 1/2. Simúlalo: representa cada carta como una lista de dos colores, elige carta y cara al azar, y cuenta solo los casos con cara roja arriba.

In [ ]:
# Escribe tu código aquí

### Ejercicio 4 (intermedio) · Monty Hall con 4 puertas
Hay 4 puertas y un carro. Eliges una, el presentador (que sabe) abre **una** puerta con cabra y te deja cambiar a **cualquiera** de las otras dos cerradas.
a) Simula la estrategia de quedarte y la de cambiar a una de las otras dos al azar.
b) ¿Cuál es mejor? Compara con 1/4 y 3/8.

In [ ]:
# Escribe tu código aquí

### Ejercicio 5 (avanzado) · ¿Niño o niña?
Una familia tiene dos hijos (cada uno es niño o niña con probabilidad 1/2, de forma independiente).
a) Si sabes que **al menos uno es niña**, ¿cuál es la probabilidad de que **los dos** sean niñas?
b) Si sabes que **la mayor es niña**, ¿cuál es la probabilidad de que los dos sean niñas?
Simula 100 000 familias y cuenta solo las que cumplen cada condición. ¿Por qué las respuestas son distintas? Relaciónalo con el Monty "despistado".

In [ ]:
# Escribe tu código aquí

---
## 🏆 Reto final (opcional) · El juego de Penney

Dos jugadores eligen cada uno una secuencia de **tres** lanzamientos de moneda, por ejemplo `CCS` y `SCC`. Se lanza una moneda una y otra vez hasta que aparece una de las dos secuencias; gana quien la eligió.

1. Simula el juego `CCS` contra `SCC`. ¿Es justo?
2. Programa una tabla con las 8 secuencias posibles enfrentadas entre sí. 
3. Descubre la paradoja: para **cualquier** secuencia que elija tu rival, hay otra que le gana con probabilidad de al menos 2/3. Es decir, ¡no existe una "mejor" secuencia! Encuentra la regla para elegir la respuesta ganadora. (Pista: mira la segunda letra de la secuencia del rival.)

In [ ]:
# Escribe tu código aquí

---
## 🎯 Cierre: lo que aprendiste hoy

- La probabilidad clásica es **casos favorables / casos posibles**; la simulación la comprueba gracias a la ley de los grandes números.
- En **Monty Hall** conviene **cambiar** (gana 2/3), porque el presentador **sabe** y su elección da información. Si abre al azar, da igual (1/2).
- Con muchas puertas la intuición se aclara: cambiar gana (n − 1)/n.
- En la **paradoja del cumpleaños**, con 23 personas ya hay más de 50 % de coincidencia, porque hay 253 parejas. Usar el **complemento** facilita el cálculo.
- En Python aprendiste: `random.Random(semilla)` para juegos reproducibles, `Dropdown` y `ToggleButtons`, `set` para detectar repetidos y `None` para marcar casos que no cuentan.

**Para pensar:** si ahora alguien te dice "conviene quedarse, ya da igual porque quedan dos puertas", ¿cómo lo convencerías? ¿Qué argumento crees que funciona mejor: la tabla, la simulación o las 100 puertas?